In [ ]:
from pathlib import Path
from dataclasses import dataclass
from functools import cached_property

import numpy as np
import numpy.typing as npt
from scipy.signal import correlate

from mbloodmoon.mask import _bisect_interval, _fold
from mbloodmoon.types import BinsRectangular, UpscaleFactor
from mbloodmoon.io import MaskDataLoader
import mbloodmoon as bm


def _bins(
    start: float,
    stop: float,
    px_size: float,
    upscaling: int,
) -> npt.NDArray:
    """
    Returns equally spaced points between start and stop, included.
    The input `start`, `stop` and `px_size` must have same dimension.

    Args:
        start (float): Start point.
        stop (float): Stop point.
        px_size (float): Size of the pixels.
        upscaling (int): Upscaling factor.

    Returns:
        output (npt.NDArray): Bin edges array.
    """
    return np.linspace(start, stop, int((stop - start) * upscaling / px_size) + 1)





@dataclass(frozen=True)
class CodedMaskCamera:
    """Dataclass containing a coded mask camera system.

    Handles mask pattern, detector geometry, and related calculations for coded mask imaging.

    Args:
        mdl: Mask data loader object containing mask and detector specifications
        upscale_f: Tuple of upscaling factors for x and y dimensions

    Raises:
        ValueError: If detector plane is larger than mask or if upscale factors are not positive
    """

    mdl: MaskDataLoader
    upscale_f: UpscaleFactor

    def _bins_mask(
        self,
        upscale_f: UpscaleFactor,
    ) -> BinsRectangular:
        """Generate binning structure for mask with given upscale factors."""
        return BinsRectangular(
            _bins(self.mdl["mask_minx"], self.mdl["mask_maxx"], self.mdl["mask_deltax"], upscale_f.x),
            _bins(self.mdl["mask_miny"], self.mdl["mask_maxy"], self.mdl["mask_deltay"], upscale_f.y),
        )
    
    def _bins_detector(
        self,
        upscale_f: UpscaleFactor,
    ) -> BinsRectangular:
        """Generate binning structure for detector with given upscale factors."""
        mask_bins = self._bins_mask(upscale_f)
        xmin, xmax = _bisect_interval(mask_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
        ymin, ymax = _bisect_interval(mask_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
        return BinsRectangular(
            mask_bins.x[xmin : xmax + 1],
            mask_bins.y[ymin : ymax + 1],
        )
    
    def _bins_sky(
        self,
        upscale_f: UpscaleFactor,
    ) -> BinsRectangular:
        """Binning structure for the reconstructed sky image."""
        pass

    @property
    def specs(self) -> dict:
        """Returns a dictionary of mask parameters useful for image reconstruction."""
        return self.mdl.specs
    
    @cached_property
    def bins_mask(self) -> BinsRectangular:
        """Binning structure for the mask pattern."""
        return self._bins_mask(self.upscale_f)

    @cached_property
    def bins_detector(self) -> BinsRectangular:
        """Binning structure for the detector."""
        return self._bins_detector(self.upscale_f)
    
    @cached_property
    def bins_sky(self) -> BinsRectangular:
        """Returns bins for the sky-shift domain"""
        return self._bins_sky(self.upscale_f)
    
    @cached_property
    def mask_shape(self) -> tuple[int, int]:
        """Shape of the mask array (rows, columns)."""
        bins = self.bins_mask
        return len(bins.y) - 1, len(bins.x) - 1
    
    @cached_property
    def detector_shape(self) -> tuple[int, int]:
        """Shape of the detector array (rows, columns)."""
        bins = self.bins_detector
        return len(bins.y) - 1, len(bins.x) - 1
    
    @cached_property
    def sky_shape(self) -> tuple[int, int]:
        """Shape of the reconstructed sky image (rows, columns)."""
        n, m = self.mask_shape
        u, v = self.detector_shape
        return n + u - 1, m + v - 1




def codedmask(
    mask_filepath: str | Path,
    upscale_x: int = 1,
    upscale_y: int = 1,
) -> CodedMaskCamera:
    """
    An interface to CodedMaskCamera.

    Args:
        mask_filepath: a str or a path object pointing to the mask filepath
        upscale_x: upscaling factor over the x direction
        upscale_y: upscaling factor over the y direction

    Returns:
        a CodedMaskCamera object.

    Raises:
        ValueError: if physical detector plane is larger than mask.
        ValueError: if upscale factors are not positive integers.
    """
    mdl = MaskDataLoader(mask_filepath)

    if not (
        # fmt: off
        mdl["detector_minx"] >= mdl["mask_minx"] and
        mdl["detector_maxx"] <= mdl["mask_maxx"] and
        mdl["detector_miny"] >= mdl["mask_miny"] and
        mdl["detector_maxy"] <= mdl["mask_maxy"]
        # fmt: on
    ):
        raise ValueError("Detector plane is larger than mask.")

    if not ((isinstance(upscale_x, int) and upscale_x > 0) and (isinstance(upscale_y, int) and upscale_y > 0)):
        raise ValueError("Upscale factors must be positive integers.")

    return CodedMaskCamera(mdl, UpscaleFactor(x=upscale_x, y=upscale_y))

In [3]:
mask_path = "/mnt/d/PhD_AASS/Coding/Images_fits/wfm_mask.fits"
wfm = codedmask(mask_path)


start_x, stop_x = wfm.specs["mask_minx"], wfm.specs["mask_maxx"]
px_size_x = wfm.specs["mask_deltax"]
upscaling_x = 3
mask_bins_x = _bins(start_x, stop_x, px_size_x, upscaling_x)

xmin, xmax = _bisect_interval(mask_bins_x, wfm.specs["detector_minx"], wfm.specs["detector_maxx"])
detector_bins_from_mask_x = mask_bins_x[xmin : xmax + 1]
detector_bins_x = _bins(mask_bins_x[xmin], mask_bins_x[xmax], wfm.specs["mask_deltax"], upscaling_x)

print(
    f"Mask edges: {start_x, stop_x}\n"
    f"Detector edges: {wfm.specs["detector_minx"], wfm.specs["detector_maxx"]}\n"
    f"mask bin [{xmin}], mask bin [{xmax}]: {mask_bins_x[xmin], mask_bins_x[xmax]}\n"
    f"xmin: {xmin}, xmax: {xmax}, len(mask_bins) - xmin: {len(mask_bins_x) - xmin}\n"
    #f"{np.all(np.abs(detector_bins_from_mask_x - detector_bins_x) < 1e-5)}\n"
    f"mask_bins_x step: {mask_bins_x[1] - mask_bins_x[0]}\n"
    f"detector_bins_from_mask_x step: {detector_bins_from_mask_x[1] - detector_bins_from_mask_x[0]}\n"
    f"detector_bins_x step: {detector_bins_x[1] - detector_bins_x[0]}\n"
)



start_y, stop_y = wfm.specs["mask_miny"], wfm.specs["mask_maxy"]
px_size_y = wfm.specs["mask_deltay"]
upscaling_y = 3
mask_bins_y = _bins(start_y, stop_y, px_size_y, upscaling_y)

ymin, ymax = _bisect_interval(mask_bins_y, wfm.specs["detector_miny"], wfm.specs["detector_maxy"])
detector_bins_from_mask_y = mask_bins_y[ymin : ymax + 1]
detector_bins_y = _bins(mask_bins_y[ymin], mask_bins_y[ymax], wfm.specs["mask_deltay"], upscaling_y)

print(
    f"Mask edges: {start_y, stop_y}\n"
    f"Detector edges: {wfm.specs["detector_miny"], wfm.specs["detector_maxy"]}\n"
    f"mask bin [{ymin}], mask bin [{ymax}]: {mask_bins_y[ymin], mask_bins_y[ymax]}\n"
    f"ymin: {ymin}, ymax: {ymax}, len(mask_bins) - xmin: {len(mask_bins_y) - ymin}\n"
    #f"{np.all(np.abs(detector_bins_from_mask_y - detector_bins_y) < 1e-5)}\n"
    f"mask_bins_y step: {mask_bins_y[1] - mask_bins_y[0]}\n"
    f"detector_bins_from_mask_y step: {detector_bins_from_mask_y[1] - detector_bins_from_mask_y[0]}\n"
    f"detector_bins_y step: {detector_bins_y[1] - detector_bins_y[0]}\n"
)


Mask edges: (-130.0, 130.0)
Detector edges: (-78.988, 78.988)
mask bin [612], mask bin [2508]: (np.float64(-79.0), np.float64(79.0))
xmin: 612, xmax: 2508, len(mask_bins) - xmin: 2509
mask_bins_x step: 0.08333333333334281
detector_bins_from_mask_x step: 0.0833333333333286
detector_bins_x step: 0.0833333333333286

Mask edges: (-130.0, 130.0)
Detector edges: (-76.5255, 76.5255)
mask bin [401], mask bin [1549]: (np.float64(-76.53333333333333), np.float64(76.53333333333333))
ymin: 401, ymax: 1549, len(mask_bins) - xmin: 1550
mask_bins_y step: 0.13333333333332575
detector_bins_from_mask_y step: 0.13333333333332575
detector_bins_y step: 0.13333333333333997



In [ ]:
def _bins_sky(
    self,
    upscale_f: UpscaleFactor,
) -> BinsRectangular:
    """Binning structure for the reconstructed sky image."""
    binsd, binsm = self._bins_detector(upscale_f), self._bins_mask(upscale_f)
    xstep, ystep = (
        binsm.x[1] - binsm.x[0],
        binsm.y[1] - binsm.y[0],
    )
    return BinsRectangular(
        np.linspace(binsd.x[0] + binsm.x[0] + xstep, binsd.x[-1] + binsm.x[-1], self.sky_shape[1] + 1),
        np.linspace(binsd.y[0] + binsm.y[0] + ystep, binsd.y[-1] + binsm.y[-1], self.sky_shape[0] + 1),
    )


def _bins_sky(
    camera,
    upscale_f: UpscaleFactor,
) -> BinsRectangular:
    """Binning structure for the reconstructed sky image."""
    det_bins, mask_bins = camera.bins_detector, camera.bins_mask
    px_x, px_y = camera.specs["mask_deltax"], camera.specs["mask_deltay"]
    return BinsRectangular(
        _bins(start, stop, px_x, upscale_f.x),
        _bins(start, stop, px_y, upscale_f.y),
    )
# TODO: assert that sky and mask binning are allined w a given tolerance


def bins_sky(camera, upscaling):
    return _bins_sky(camera, upscaling)


def sky_shape(camera, upscaling) -> tuple[int, int]:
    """Shape of the reconstructed sky image (rows, columns)."""
    bins = bins_sky(upscaling)
    n, m = camera.mask_shape
    u, v = camera.detector_shape
    assert (len(bins.y) - 1, len(bins.x) - 1) == (n + u - 1, m + v - 1)
    return len(bins.y) - 1, len(bins.x) - 1

In [3]:
def oversample(data, upscaling):
    if isinstance(upscaling, int): upscaling = (upscaling,)
    for i, f in enumerate(upscaling):
        data = np.repeat(data, f, axis=i)
    return data/np.prod(upscaling)


def _print_info(start, stop, px_size, upscaling):
    bins = _bins(start, stop, px_size, upscaling)
    px_struct = np.ones(int((stop - start) / px_size))
    print(
        f"Upscaling: {upscaling}\n"
        f"# divs: {int((stop - start) * upscaling / px_size)}\n"
        f"Binning length: {len(bins)}\n"
        f"# of pixels: {len(bins) - 1}\n"
        f"Oversampled data length: {len(oversample(px_struct, upscaling))}\n"
    )


mask_path = "/mnt/d/PhD_AASS/Coding/Images_fits/wfm_mask.fits"
wfm = bm.codedmask(mask_path, upscale_x=1, upscale_y=1)

start, stop = wfm.specs["mask_minx"], wfm.specs["mask_maxx"]
px_size = wfm.specs["mask_deltax"]

_print_info(start, stop, px_size, upscaling=1)
_print_info(start, stop, px_size, upscaling=2)
_print_info(start, stop, px_size, upscaling=3)
_print_info(start, stop, px_size, upscaling=4)

Upscaling: 1
# divs: 1040
Binning length: 1041
# of pixels: 1040
Oversampled data length: 1040

Upscaling: 2
# divs: 2080
Binning length: 2081
# of pixels: 2080
Oversampled data length: 2080

Upscaling: 3
# divs: 3120
Binning length: 3121
# of pixels: 3120
Oversampled data length: 3120

Upscaling: 4
# divs: 4160
Binning length: 4161
# of pixels: 4160
Oversampled data length: 4160



In [ ]:
import sys
import tempfile

import numpy as np
import numpy.typing as npt
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_pixel
from astropy.wcs.wcsapi import SlicedLowLevelWCS

from reproject.array_utils import iterate_chunks, sample_array_edges
from reproject.utils import parse_input_data, parse_output_projection
from reproject.mosaicking.subset_array import ReprojectedArraySubset

IS_WIN = sys.platform == "win32"


def sky_composition(
    input_data,
    output_projection,
    shape_out,
    reproject_function,
    combine_function,
) -> tuple[np.array, np.array]:
    
    # Parse the output projection to avoid having to do it for each
    wcs_out, shape_out = parse_output_projection(output_projection, shape_out=shape_out)

    output_array = np.zeros(shape_out)

    output_footprint = np.zeros(shape_out)

    on_the_fly = combine_function in ("mean", "sum")


    # Start off by reprojecting individual images to the final projection
    if not on_the_fly:
        arrays = []

    with tempfile.TemporaryDirectory(ignore_cleanup_errors=IS_WIN) as local_tmp_dir:
        for idata in range(len(input_data)):
            # We need to pre-parse the data here since we need to figure out how to
            # optimize/minimize the size of each output tile (see below).
            array_in, wcs_in = parse_input_data(input_data[idata], hdu_in=None)

            # Since we might be reprojecting small images into a large mosaic we
            # want to make sure that for each image we reproject to an array with
            # minimal footprint. We therefore find the pixel coordinates of the
            # edges of the initial image and transform this to pixel coordinates in
            # the final image to figure out the final WCS and shape to reproject to
            # for each tile. We strike a balance between transforming only the
            # input-image corners, which is fast but can cause clipping in cases of
            # significant distortion (when the edges of the input image become
            # convex in the output projection), and transforming every edge pixel,
            # which provides a lot of redundant information.
            edges = sample_array_edges(array_in.shape, n_samples=11)[::-1]
            edges_out = pixel_to_pixel(wcs_in, wcs_out, *edges)[::-1]

            # Determine the cutout parameters

            # In some cases, images might not have valid coordinates in the corners,
            # such as all-sky images or full solar disk views. In this case we skip
            # this step and just use the full output WCS for reprojection.
            ndim_out = len(shape_out)
            if np.any(np.isnan(edges_out)):
                bounds = list(zip([0] * ndim_out, shape_out, strict=False))
            else:
                bounds = []
                for idim in range(ndim_out):
                    imin = max(0, int(np.floor(edges_out[idim].min() + 0.5)))
                    imax = min(shape_out[idim], int(np.ceil(edges_out[idim].max() + 0.5)))
                    bounds.append((imin, imax))
                    if imax < imin: break

            slice_out = tuple([slice(imin, imax) for (imin, imax) in bounds])

            if isinstance(wcs_out, WCS):
                wcs_out_indiv = wcs_out[slice_out]
            else:
                wcs_out_indiv = SlicedLowLevelWCS(wcs_out.low_level_wcs, slice_out)

            shape_out_indiv = tuple([imax - imin for (imin, imax) in bounds])

            array = footprint = None

            array, footprint = reproject_function(
                (array_in, wcs_in),
                output_projection=wcs_out_indiv,
                shape_out=shape_out_indiv,
                hdu_in=None,
                output_array=array,
                output_footprint=footprint,
            )

            # For the purposes of mosaicking, we mask out NaN values from the array
            # and set the footprint to 0 at these locations.
            reset = np.isnan(array)
            array[reset] = 0.0
            footprint[reset] = 0.0

            array = ReprojectedArraySubset(array, footprint, bounds)

            if on_the_fly:
                # By default, values outside of the footprint are set to NaN
                # but we set these to 0 here to avoid getting NaNs in the
                # means/sums.
                array.array[array.footprint == 0] = 0
                output_footprint[array.view_in_original_array] += array.footprint
                # We now need to do output[view] += array * footprint but to avoid
                # the temporary array allocation from array * footprint we modify
                # array inplace, which we can do as the array will be discarded at
                # the end of the loop.
                array.array *= array.footprint
                output_array[array.view_in_original_array] += array.array

            else:
                arrays.append(array)


        if combine_function == "mean":
            with np.errstate(invalid="ignore"):
                output_array /= output_footprint

        if combine_function in ("first", "last", "min", "max"):
            if combine_function == "min":
                output_array[...] = np.inf
            elif combine_function == "max":
                output_array[...] = -np.inf

            for array in arrays:
                if combine_function == "first":
                    mask = output_footprint[array.view_in_original_array] == 0
                elif combine_function == "last":
                    mask = array.footprint > 0
                elif combine_function == "min":
                    mask = (array.footprint > 0) & (
                        array.array < output_array[array.view_in_original_array]
                    )
                elif combine_function == "max":
                    mask = (array.footprint > 0) & (
                        array.array > output_array[array.view_in_original_array]
                    )

                output_footprint[array.view_in_original_array] = np.where(
                    mask, array.footprint, output_footprint[array.view_in_original_array]
                )
                output_array[array.view_in_original_array] = np.where(
                    mask, array.array, output_array[array.view_in_original_array]
                )

    # We need to avoid potentially large memory allocation from output == 0 so
    # we operate in chunks.
    for chunk in iterate_chunks(output_array.shape, max_chunk_size=256 * 1024**2):
        output_array[chunk][output_footprint[chunk] == 0] = 0

    return output_array, output_footprint